# SimPipeline quickstart

Runs the full loop: simulate users → stream events → compute features → train the two-tower model → evaluate offline (held-out Recall@K) → deploy → watch online CTR.

**Locally or in a cloud session:** open this notebook from the repo root after `pip install -e ".[dev]"`
**On Colab:** just run the cells. The setup cell clones the repo for you (it's public, so no token is needed).

Every run with the same `SEED` gives the same events and the same model.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# On Colab, fetch the repo and work inside it. Elsewhere, run from the repo root.
if "google.colab" in sys.modules:
    repo = Path("/content/SimPipelineExp")
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/PrimozRavbar/SimPipelineExp.git", str(repo)],
            check=True,
        )
    os.chdir(repo)

ROOT = Path.cwd()
assert (ROOT / "ml_system").is_dir(), f"Open this notebook from the repo root (currently {ROOT})"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Repo:", ROOT)

## Settings

In [ ]:
SEED = 42                 # same seed -> same simulation and same model
SIM_TICKS = 5000          # one simulated user session per tick
POST_RETRAIN_TICKS = 3000 # ticks served by the retrained model
EPOCHS = 5
NEGATIVE_SAMPLES = 5
SLATE_SIZE = 10           # items shown per session (cut from 100 retrieved candidates)
K = 20                    # Recall@K
CTR_WINDOW = 250          # ticks in the rolling CTR window

## Load MovieLens and build the system

In [ ]:
from world.external_data_loader import load_movielens
from ml_system.build_recommendation_system import build_recommendation_system

world_data = load_movielens()   # uses the ml-100k folder in the repo

system = build_recommendation_system(
    movie_objects=world_data["movie_objects"],
    user_objects=world_data["user_objects"],
    user_states=world_data["user_states"],
    movie_states=world_data["movie_states"],
    seed=SEED,
    slate_size=SLATE_SIZE,
)

print("Users:", len(world_data["user_objects"]), "| Movies:", len(world_data["movie_objects"]))

## Run the simulation

In [ ]:
from collections import Counter

system["sim"].run(SIM_TICKS)

events = system["datalake"].events
ctr_tracker = system["ctr_tracker"]

print("DataLake events:", len(events))
print(Counter(type(event).__name__ for event in events))
print(f"CTR with the bootstrap model: {ctr_tracker.overall_ctr():.1%}")

## Offline features (Spark)

In [ ]:
spark = system["spark"]

spark.run_offline_feature_job(events, system["offline_pipeline"].feature_pipeline)

job = spark.application.jobs[0]
print("Application:", spark.application.state.value, spark.application.metrics)
print("Job:", job.state.value, "| Stage:", job.stages[0].state.value, "| Task:", job.stages[0].tasks[0].state.value)

## Train the two-tower model

Training negatives are the items the user was shown and skipped, never retrieved-but-unshown candidates.

In [ ]:
from ml_system.retrain import retrain

system["offline_pipeline"].dataset_generator.negative_samples = NEGATIVE_SAMPLES
system["offline_pipeline"].trainer.epochs = EPOCHS

artifacts = retrain(system)

# Freeze the training-period events: the DataLake list keeps growing below.
retrain_tick = system["sim"].clock.now
training_period_events = list(events)

## Offline: held-out Recall@K

`RetrievalEvaluator` trains a fresh two-tower model on the first 80% of the training-period events and scores Recall@K on the last 20%. It compares the result against popularity and random baselines. Users who appear only in the held-out part are skipped, because they are cold-start.

Caveat: `evaluate_popularity` currently pads its list with held-out movies, so it is optimistic (see CLAUDE.md known issues).

In [ ]:
from ml_system.evaluation import RetrievalEvaluator, evaluate_popularity, evaluate_random

evaluator = RetrievalEvaluator(
    world_data["movie_objects"],
    num_users=len(world_data["user_objects"]),
    k=K,
    seed=SEED,
    epochs=EPOCHS,
)
held_out = evaluator.evaluate(training_period_events)

population = held_out["validation_population"]
popularity = evaluate_popularity(training_period_events, population, k=K)
random_baseline = evaluate_random(population, world_data["movie_objects"], k=K, seed=SEED)

print(f"Held-out interactions scored: {held_out['evaluated']}")
for name, result in [("two-tower", held_out), ("popularity", popularity), ("random", random_baseline)]:
    print(f"  Recall@{K} {name:>10}: {result['recall_at_k']:.2%}")

## Online: deploy the retrained model and watch CTR

`retrain()` already swapped the new towers and index into the live retriever. Run more ticks and compare click-through rate before and after. As reference points from CLAUDE.md, a random slate gets about 18% CTR and an oracle slate about 68%.

In [ ]:
system["sim"].run(POST_RETRAIN_TICKS)

before = ctr_tracker.ctr_between(None, retrain_tick)
after = ctr_tracker.ctr_between(retrain_tick + 1, None)

print(f"CTR before retrain (ticks 1-{retrain_tick}): {before:.1%}")
print(f"CTR after retrain  (ticks {retrain_tick + 1}-{system['sim'].clock.now}): {after:.1%}")

In [ ]:
import matplotlib.pyplot as plt

# Skip the first CTR_WINDOW ticks, while the window is still filling up.
series = ctr_tracker.rolling_ctr(window=CTR_WINDOW)[CTR_WINDOW - 1:]
ticks = [tick for tick, ctr in series]
values = [ctr for tick, ctr in series]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(ticks, values, linewidth=1.5, label=f"rolling CTR ({CTR_WINDOW} ticks)")
ax.axhline(0.18, color="gray", linestyle=":", linewidth=1, label="random slate (~18%)")
ax.axvline(retrain_tick, color="gray", linestyle="--", linewidth=1, label="retrain + deploy")
ax.set_xlabel("tick")
ax.set_ylabel("CTR")
ax.set_ylim(0, max(max(v for v in values if v is not None), 0.18) * 1.3)
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.legend(loc="lower right", frameon=False)
ax.set_title("Online click-through rate")
plt.tight_layout()
plt.show()